# Serial-v0 interface 살펴보기

원본 `gym-invmgmt`의 `reset` / `step`이 무엇을 돌려주는지 보는 노트북입니다.

- **원본 interface**: `gym.make`, `reset`, `step`이 반환하는 `obs`, `reward`, `terminated`, `truncated`, `info`
- **우리가 추가한 출력**: 그 값을 STATUS / ACTION / REWARD로 풀어 쓰는 부분

환경은 과제에 맞게 `GymInvMgmt/Serial-v0` + `use_goodwill=True`입니다.

사용 순서:
1. 설정 셀을 실행합니다.
2. `reset` 셀을 실행합니다.
3. `ACTION` 숫자를 바꾸고 `step` 셀을 실행합니다.
4. 같은 에피소드를 이어가려면 `step`만 다시 실행합니다.
5. 처음부터 보려면 `reset`을 다시 실행합니다.

In [ ]:
from pathlib import Path
import sys

import gymnasium as gym
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

import gym_invmgmt  # noqa: F401
from walk_serial_steps import SERIAL_DEMAND_CONFIG, print_environment, print_period

print("gym_invmgmt", gym_invmgmt.__version__)
print("프로젝트 경로", ROOT)

## 1. 환경 만들기

원본 호출은 `gym.make`입니다. `demand_config`를 넘기면 기본값을 통째로 바꿉니다.

In [ ]:
env = gym.make("GymInvMgmt/Serial-v0", demand_config=SERIAL_DEMAND_CONFIG)
core = env.unwrapped

print("환경 ID:", env.spec.id)
print("관측 공간:", env.observation_space)
print("행동 공간:", env.action_space)
print("행동 하한:", env.action_space.low)
print("행동 상한:", env.action_space.high)
print("행동 의미:")
print("  action[0]  유통 2 → 소매 1")
print("  action[1]  공장 3 → 유통 2")
print("  action[2]  원자재 4 → 공장 3")
print()
print_environment(core)

## 2. reset

원본은 `(obs, info)`를 돌려줍니다. 에피소드를 처음부터 다시 보려면 이 셀만 다시 실행하면 됩니다.

In [ ]:
SEED = 42
obs, info = env.reset(seed=SEED)

print("원본 reset 반환")
print("  obs 타입:", type(obs), "shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  info =", info)
print("  현재 기간 core.period =", core.period)

In [ ]:
def decode_obs(obs):
    """원본 관측 벡터를 칸별로 풀어 본다. 학습에는 이 함수가 필요 없다."""
    obs = np.asarray(obs, dtype=float).reshape(-1)
    return {
        "직전 수요": obs[0],
        "직전 미충족": obs[1],
        "재고 소매": obs[2],
        "재고 유통": obs[3],
        "재고 공장": obs[4],
        "파이프라인 2→1 (0~3기간 후)": obs[5:9],
        "파이프라인 3→2 (0~3기간 후)": obs[9:13],
        "기간 t": obs[13],
        "sentiment": obs[14],
    }

print("우리가 풀어 본 현재 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")

## 3. action 넣고 step

아래 숫자만 바꾼 뒤 `step` 셀을 실행합니다.

- `action[0]`: 유통 → 소매
- `action[1]`: 공장 → 유통
- `action[2]`: 원자재 → 공장

In [ ]:
ACTION = [20.0, 20.0, 20.0]
action = np.asarray(ACTION, dtype=np.float64)
print("이번에 넣을 action =", action)

In [ ]:
period_before = core.period
obs, reward, terminated, truncated, info = env.step(action)

print("원본 step 반환")
print("  obs shape:", np.shape(obs))
print("  obs =", np.asarray(obs, dtype=float))
print("  reward =", float(reward))
print("  terminated =", terminated)
print("  truncated =", truncated)
print("  info =", info)
print()
print("우리가 풀어 본 다음 STATUS")
for key, value in decode_obs(obs).items():
    print(f"  {key}: {value}")
print()
print_period(core, period_before, action, float(reward), info)

## 사용 방법

- 다른 발주량을 보려면 `ACTION = [10, 30, 20]`처럼 바꾸고 `ACTION` 셀과 `step` 셀을 다시 실행합니다.
- 30기간이 끝나면 `truncated=True`가 됩니다. 그때는 `reset`부터 다시 실행합니다.
- 학습 코드가 쓰는 값은 `obs`, `reward`, `terminated`, `truncated`입니다. 아래 STATUS/REWARD 산식은 이해를 위한 출력입니다.